[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbanuelos/grad_math_modeling/blob/main/Lectures/Module3_DeepLearning/12_RandomForests.ipynb#copy=true)


# Random Forests: Ensembles of Decision Trees

## Learning objectives

By the end of this lesson, you should be able to:

- explain bootstrap sampling and random feature selection in a random forest;
- compare a single decision tree with an ensemble on held-out data;
- interpret out-of-bag estimates and feature importance cautiously;
- describe why averaging less-correlated trees can reduce variance.

**Student Learning Outcome:** Compare nonlinear models using a held-out evaluation and interpret diagnostics in context.


## From one tree to many

A single tree can change substantially when its training data change slightly. A **random forest** fits many trees to bootstrap samples: samples drawn with replacement from the training data. At each split, each tree considers only a random subset of features. The forest predicts the class receiving the most votes.

These sources of randomness make individual trees less alike. Voting across less-correlated trees can reduce variance while retaining nonlinear boundaries.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

RANDOM_STATE = 232
wine = load_wine(as_frame=True)
X, y = wine.data, wine.target
class_names = wine.target_names
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
print("features:", list(X.columns))
print("training observations:", len(X_train), "test observations:", len(X_test))


## Part 1 — Baseline tree and forest

Use the same train/test split for both models so their test scores are comparable. A test score estimates performance on this held-out set; it is not a guarantee of future performance.


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

single_tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
forest = RandomForestClassifier(n_estimators=300, max_features="sqrt", oob_score=True,
                                random_state=RANDOM_STATE, n_jobs=-1)
for name, model in [("single tree", single_tree), ("random forest", forest)]:
    model.fit(X_train, y_train)
    print(f"{name:>14}: train={model.score(X_train, y_train):.3f}, test={model.score(X_test, y_test):.3f}")
print("forest out-of-bag accuracy:", round(forest.oob_score_, 3))


### What is out-of-bag (OOB) accuracy?

Each bootstrap sample leaves out some training observations. For an observation, the trees that did not train on it can vote on its class. Combining those votes gives an **out-of-bag estimate**, an internal validation-style diagnostic. It is useful, but it does not replace a final test set.


### Checkpoint 1

Why does a forest use both bootstrap samples and random feature subsets? Explain why fitting 300 identical trees to the same data would not provide the same benefit.

> **YOUR ANSWER:**


## Part 2 — Feature importance is not causation

`feature_importances_` reports how much splits using a feature reduced impurity across the trees. A large value can indicate usefulness to this fitted forest, but correlated features can share or mask importance, and importance does not establish a causal effect.


In [ ]:
importance = pd.Series(forest.feature_importances_, index=X.columns).sort_values()
importance.plot.barh(figsize=(8, 5), color="tab:green")
plt.xlabel("mean decrease in impurity")
plt.title("Random-forest impurity-based feature importance")
plt.show()


### Practice 2

Name the two most important displayed features. Give one reason this ranking might change with a new sample of wines or a strongly correlated measurement.

> **YOUR ANSWER:**


## Part 3 — Number of trees and stability

Adding trees generally stabilizes a forest's vote; it does not authorize unlimited reuse of test data. Explore the training workflow below, then reserve a final claim for a held-out evaluation.


In [ ]:
rows = []
for count in [1, 5, 20, 100, 300]:
    candidate = RandomForestClassifier(n_estimators=count, max_features="sqrt",
                                       random_state=RANDOM_STATE, n_jobs=-1).fit(X_train, y_train)
    rows.append({"trees": count, "training accuracy": candidate.score(X_train, y_train),
                 "test accuracy": candidate.score(X_test, y_test)})
pd.DataFrame(rows).round(3)


## Lesson summary

Random forests trade some single-tree interpretability for more stable predictions. Scores, feature-importance plots, and OOB estimates still require careful interpretation. Next, boosting will build an ensemble sequentially rather than averaging independently grown trees.
